# Ejercicios UD1 · Bloque 1 — Naturaleza, tipos de datos, entidades y carga

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)
**Unidad Didáctica:** UD1 — Análisis exploratorio de datos: estructura, variables y calidad
**Alumno/a:**
**Fecha de entrega:**

---

Este cuaderno recoge los ejercicios del **Bloque 1** de la UD1. Están organizados por sesiones, igual que los cuadernos de teoría:

| Sesión | Cuaderno de teoría | Ejercicios |
|---|---|---|
| 1 | `B1_S1_Naturaleza_y_Tipos_de_Datos` | 1.1 – 1.5 |
| 2 | `B1_S2_Identificacion_de_Entidades_y_Relaciones` | 2.1 – 2.3 |
| 3 | `B1_S3_Formatos_Carga_y_Repositorios` | 3.1 – 3.3 |

Algunos ejercicios son preguntas de reflexión (responde en la celda de texto), otros son de código (escribe tu solución donde pone `TODO`) y otros te piden que busques información por tu cuenta.

> **Antes de empezar:** copia este cuaderno a tu carpeta `mis_cuadernos/` y trabaja sobre la copia, así `git pull` no te dará conflictos.
>
> **Cómo entregar:** reinicia el kernel y ejecuta todas las celdas (**Restart** → **Run All**), comprueba que no hay errores, guarda y sube el fichero `.ipynb` al aula virtual.

## Criterios de evaluación

Esta relación de ejercicios evalúa los siguientes criterios de evaluación del módulo (Real Decreto 183/2026, redacción literal):

> **RA1.** Identifica la estructura y variables en los datos en el dominio de aplicación, así como su cobertura y calidad del conjunto, aplicando técnicas exploratorias mediante análisis exploratorios de datos (AED) para la selección de atributos clave en el entrenamiento posterior.

| CE | Criterio de evaluación | Ejercicios |
|:--:|------------------------|:----------:|
| **1.a** | Se han descrito técnicas y posibilidades de almacenaje de los datos para componer conjuntos, identificando su naturaleza. | 1.1, 1.2, 1.5, 3.1, 3.2, 3.3 |
| **1.b** | Se han detectado datos incorrectamente introducidos, ausencia de valores, una mala codificación de las variables o valores atípicos. | 1.3, 1.4, 2.1, 2.2, 3.1, 3.2 |
| **1.c** | Se han identificado la estructura, variables y relaciones de los conjuntos de datos aplicando técnicas exploratorias. | 1.2, 1.3, 1.4, 1.5, 2.1 |
| **1.d** | Se han identificado entidades, volumetrías, relaciones y atributos, para documentar cada conjunto de datos, describiendo sus características. | 2.1, 2.2, 2.3, 3.2 |

In [ ]:
# Librerías que se usan en todo el cuaderno
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

print('✅ Librerías cargadas')

---
# Sesión 1 — Naturaleza y tipos de datos

## Ejercicio 1.1 — ¿Qué naturaleza tienen estos datos?

Para cada fuente de datos, indica:

- Si los datos son **estructurados**, **semiestructurados** o **no estructurados**.
- Una justificación breve.
- **Qué harías para convertirlos en una tabla** (filas × columnas) con la que entrenar un modelo. Por ejemplo: «una fila por …, con las columnas …».

| # | Fuente de datos | Naturaleza | Justificación | ¿Cómo la convertirías en tabla? |
|---|-----------------|------------|---------------|---------------------------------|
| 1 | CSV de datos abiertos del Ayuntamiento de Granada con la calidad del aire diaria por estación | | | |
| 2 | Respuesta JSON de la API de AEMET con la predicción del tiempo por municipio | | | |
| 3 | Informes de alta hospitalaria escaneados en PDF | | | |
| 4 | Fichero de log del servidor web del instituto | | | |
| 5 | Factura electrónica en formato Facturae (XML) | | | |
| 6 | Exportación a Excel de las calificaciones de un grupo desde Séneca | | | |
| 7 | Grabaciones de audio de las llamadas al 112 | | | |
| 8 | Reseñas de Google Maps de los restaurantes del Albaicín | | | |

*Edita la tabla directamente en esta celda o responde en la celda de abajo.*

*Escribe aquí tus respuestas si prefieres no editar la tabla:*

## Ejercicio 1.2 — Aplanar los pedidos de una librería

Una librería de Granada vende por internet. Su tienda online exporta los pedidos en **JSON**: cada pedido tiene los datos del cliente anidados, una **lista de líneas** (un libro por línea) y una lista de etiquetas.

Ejecuta la celda para ver los datos.

In [ ]:
pedidos_json = [
    {
        'id_pedido': 'P-1001', 'fecha': '2026-09-02',
        'cliente': {'id': 'C01', 'nombre': 'Lucía Romero', 'ciudad': 'Granada'},
        'lineas': [
            {'isbn': '978-84-376-0494-7', 'titulo': 'Cien años de soledad', 'unidades': 1, 'precio': 12.95},
            {'isbn': '978-84-206-7495-1', 'titulo': 'Romancero gitano', 'unidades': 2, 'precio': 8.50},
        ],
        'etiquetas': ['online', 'regalo'],
    },
    {
        'id_pedido': 'P-1002', 'fecha': '2026-09-03',
        'cliente': {'id': 'C02', 'nombre': 'Javier Ortega', 'ciudad': 'Armilla'},
        'lineas': [
            {'isbn': '978-84-9759-311-1', 'titulo': 'La sombra del viento', 'unidades': 1, 'precio': 10.95},
        ],
        'etiquetas': ['recogida_tienda'],
    },
    {
        'id_pedido': 'P-1003', 'fecha': '2026-09-03',
        'cliente': {'id': 'C01', 'nombre': 'Lucía Romero', 'ciudad': 'Granada'},
        'lineas': [
            {'isbn': '978-84-206-7495-1', 'titulo': 'Romancero gitano', 'unidades': 1, 'precio': 8.50},
            {'isbn': '978-84-670-3449-2', 'titulo': 'El infinito en un junco', 'unidades': 1, 'precio': 22.90},
            {'isbn': '978-84-376-0494-7', 'titulo': 'Cien años de soledad', 'unidades': 1, 'precio': 12.95},
        ],
        'etiquetas': ['online'],
    },
    {
        'id_pedido': 'P-1004', 'fecha': '2026-09-05',
        'cliente': {'id': 'C03', 'nombre': 'Rocío Medina', 'ciudad': 'Motril'},
        'lineas': [
            {'isbn': '978-84-670-3449-2', 'titulo': 'El infinito en un junco', 'unidades': 2, 'precio': 22.90},
        ],
        'etiquetas': ['online', 'regalo', 'envio_urgente'],
    },
]

print(json.dumps(pedidos_json[0], indent=2, ensure_ascii=False))

**Parte A — Una fila por pedido.**
Usa `pd.json_normalize()` para obtener un DataFrame con **una fila por pedido** y las columnas del cliente aplanadas (`cliente.id`, `cliente.nombre`, `cliente.ciudad`). Añade una columna `num_lineas` con el número de libros distintos de cada pedido.

In [ ]:
# TODO: una fila por pedido
# df_pedidos = pd.json_normalize(...)
# df_pedidos['num_lineas'] = ...

# df_pedidos

**Parte B — Una fila por línea de pedido.**
Usa los parámetros `record_path` y `meta` de `pd.json_normalize()` para obtener **una fila por línea** (libro dentro de un pedido), conservando `id_pedido`, `fecha` y el `id` del cliente. Calcula el importe de cada línea (`unidades × precio`) y el **importe total de cada pedido**.

In [ ]:
# TODO: una fila por línea de pedido
# df_lineas = pd.json_normalize(pedidos_json, record_path=..., meta=[...])
# df_lineas['importe'] = ...

# TODO: importe total por pedido
# total_pedido = ...

**Parte C — Etiquetas en columnas.**
Convierte la lista de `etiquetas` en columnas binarias (*multi-hot encoding*) y únelas al DataFrame de la Parte A.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

# TODO: multi-hot encoding de las etiquetas
# mlb = MultiLabelBinarizer()
# ...

**Preguntas:**

1. ¿Por qué los datos originales son **semiestructurados** y no estructurados?
2. Las tablas de las partes A y B tienen distinto número de filas. ¿Qué representa una fila en cada una? Si quisieras entrenar un modelo que prediga si un pedido es un **regalo**, ¿cuál usarías? ¿Y si quisieras predecir **qué libro** se va a vender más?
3. En la Parte B, los datos del cliente y el título del libro se repiten en varias filas. ¿Qué problema puede causar esa repetición?

*Escribe aquí tus respuestas:*

## Ejercicio 1.3 — Diamantes: variables ordinales bien codificadas

El dataset `diamonds` de seaborn tiene casi 54.000 diamantes con su precio y sus características. Tres de ellas son **ordinales**:

| Variable | Categorías, de **peor** a **mejor** |
|---|---|
| `cut` (talla) | Fair < Good < Very Good < Premium < Ideal |
| `color` | J < I < H < G < F < E < D (D es el más incoloro, el mejor) |
| `clarity` (pureza) | I1 < SI2 < SI1 < VS2 < VS1 < VVS2 < VVS1 < IF |

**Parte A — Diagnóstico.**
Carga el dataset y construye una tabla con, para cada columna: el `dtype` que asigna pandas, el número de valores distintos y su **tipo estadístico**. Comprueba también si las tres variables ordinales están marcadas como ordenadas (`.cat.ordered`) y en qué orden están sus categorías (`.cat.categories`).

In [ ]:
diamonds = sns.load_dataset('diamonds')

# TODO: tabla de diagnóstico (dtype, valores distintos, tipo estadístico)

# TODO: ¿están ordenadas cut, color y clarity? ¿en qué orden?

**Parte B — Corrección.**
Define un `pd.CategoricalDtype(..., ordered=True)` para cada variable ordinal, con las categorías **de peor a mejor**, y aplícalo. Después:

1. Cuenta cuántos diamantes tienen una talla **igual o mejor que** `Very Good`.
2. Cuenta cuántos tienen a la vez color **mejor que** `G` y pureza **igual o mejor que** `VS1`.
3. Muestra el **precio medio por talla**, ordenado de peor a mejor talla (pista: `groupby(..., observed=True)`).

In [ ]:
# TODO: tipos ordinales
# orden_cut = pd.CategoricalDtype([...], ordered=True)
# ...

# TODO: preguntas 1 a 3

**Preguntas:**

1. Seaborn ya carga estas columnas como `category`. ¿Por qué no basta con eso para responder a las preguntas 1 y 2?
2. ¿Qué pasaría si codificaras `color` con números (`D=1, E=2, …, J=7`) y entrenaras un modelo con ellos? ¿Es buena idea? ¿Qué deberías tener en cuenta al elegir los números?
3. El precio medio por talla puede dar un resultado sorprendente (las mejores tallas no tienen por qué ser las más caras). ¿Se te ocurre qué otra variable puede explicarlo?

*Escribe aquí tus respuestas:*

## Ejercicio 1.4 — La encuesta del primer día

El primer día de clase se pasó un formulario al alumnado. La exportación del formulario llega así: **todas las columnas son texto** y cada persona ha escrito las respuestas a su manera.

In [ ]:
encuesta = pd.DataFrame({
    'marca_temporal': ['28/09/2026 10:15:32', '28/09/2026 10:16:05', '28/09/2026 10:16:40',
                       '28/09/2026 10:17:12', '28/09/2026 10:18:55', '28/09/2026 10:19:03',
                       '28/09/2026 10:21:47', '28/09/2026 10:22:30'],
    'fecha_nacimiento': ['14/03/2005', '02/11/2001', '30/06/2004', '2003-01-21',
                         '19/09/1997', '07/05/2005', '', '25/12/2004'],
    'edad': ['21', '24', '', '23 años', '29', 'veintiuno', '31', '12'],
    'trabaja': ['Sí', 'No', 'No', '', 'Sí', 'sí', 'No', 'NS/NC'],
    'nivel_ingles': ['B1', 'B2', 'A2', 'C1', 'b1', 'B2', None, 'A2'],
    'nota_acceso': ['7,5', '8.25', '6', '', '9,1', '7', '5,75', '8'],
    'horas_estudio_semana': ['10', '5', '8', '12', '', '6', '3', '15'],
    'localidad': ['Granada', 'granada ', 'Armilla', 'Granada', 'Maracena', 'Motril', 'Granada', 'Santa Fe'],
})

encuesta

**Parte A — Diagnóstico.**
Para cada columna, indica su **tipo estadístico**, el **`dtype` que debería tener** y los **problemas de codificación** que ves (formatos mezclados, valores escritos de distintas formas, vacíos que deberían ser nulos…).

| Columna | Tipo estadístico | `dtype` objetivo | Problemas detectados |
|---|---|---|---|
| `marca_temporal` | | | |
| `fecha_nacimiento` | | | |
| `edad` | | | |
| `trabaja` | | | |
| `nivel_ingles` | | | |
| `nota_acceso` | | | |
| `horas_estudio_semana` | | | |
| `localidad` | | | |

**Parte B — Corrección.**
Crea `encuesta_ok` con todas las columnas corregidas. Requisitos:

- Los vacíos y las respuestas no válidas deben quedar como **nulos**, no como texto.
- `edad` y `horas_estudio_semana` deben ser enteros **que admitan nulos**.
- `trabaja` debe ser booleano **que admita nulos** (`NS/NC` y el vacío son nulos).
- `nivel_ingles` debe ser **ordinal** con los niveles del MCER: A1 < A2 < B1 < B2 < C1 < C2.
- `localidad` debe ser categórica, sin diferencias por mayúsculas o espacios.
- Las fechas deben ser `datetime64`. Ojo: la fecha de nacimiento tiene dos formatos distintos.

Pistas: `pd.to_numeric(..., errors='coerce')`, `pd.to_datetime(..., format=..., errors='coerce')`, `.str.strip()`, `.str.upper()`, `.str.replace()`, `.map()`, `astype('Int64')`, `astype('boolean')`.

In [ ]:
encuesta_ok = encuesta.copy()

# TODO: corrige cada columna

# Comprobación final
# print(encuesta_ok.dtypes)
# encuesta_ok

**Parte C — Comprobación cruzada.**
Tienes la edad por dos caminos: la que escribió cada persona y la que se puede calcular con la fecha de nacimiento y la marca temporal. Calcula la edad a partir de las fechas y compárala con la columna `edad`.

- ¿Puedes recuperar alguna edad que quedó como nula?
- ¿Hay alguna respuesta incoherente?

In [ ]:
# TODO: edad calculada a partir de las fechas y comparación

**Preguntas:**

1. ¿Por qué no se puede usar `astype('int64')` en `edad` después de convertirla? ¿Qué diferencia hay entre `int64` e `Int64`?
2. ¿Por qué `astype('bool')` no sirve para la columna `trabaja`?
3. ¿Qué cambiarías en el **formulario** para que estos problemas no se produzcan?

*Escribe aquí tus respuestas:*

## Ejercicio 1.5 — Estructurar anuncios de pisos con un LLM

Los anuncios de alquiler son **texto libre**: cada anunciante escribe a su manera. Queremos convertirlos en una tabla con estas columnas:

| Columna | Tipo |
|---|---|
| `barrio` | categórica nominal |
| `precio_mes` | numérica (euros) |
| `metros` | numérica |
| `habitaciones` | numérica discreta |
| `amueblado` | booleana (nula si el anuncio no lo dice) |
| `mascotas` | booleana (nula si el anuncio no lo dice) |

Como en el Ejemplo 3b de la sesión 1, usaremos un modelo de lenguaje a través de **OpenRouter**. Necesitas tu archivo `.env` con la clave `OPENROUTER_API_KEY`.

In [ ]:
anuncios = pd.DataFrame({
    'id': ['A1', 'A2', 'A3', 'A4', 'A5', 'A6'],
    'texto': [
        'Se alquila piso en el Zaidín, 3 dormitorios, 85 m2, amueblado. 650 €/mes. No se admiten mascotas.',
        'Precioso ático en el Realejo con vistas a la Alhambra. Dos habitaciones y unos 70 metros. 900 euros al mes, sin muebles.',
        'Estudio de 35m² junto a la Facultad de Ciencias (Fuentenueva). Ideal estudiantes. 480€. Mascotas bienvenidas 🐶',
        'Alquilo piso grande en la Chana: cuatro habitaciones, 110 metros cuadrados, totalmente amueblado. Precio: 750 al mes.',
        'Albaicín, casa con patio, 2 hab, 60 m2. 700 euros mensuales. Se aceptan gatos.',
        'Piso en Camino de Ronda, 90 m², 3 habitaciones. Consultar precio.',
    ],
})

anuncios

**Parte A — Extraer los datos.**
Completa la función `extraer_campos()` para que pida al modelo un **JSON** con los seis campos y lo devuelva como diccionario. Después aplícala a todos los anuncios y construye un DataFrame con el resultado.

Pistas:
- En las instrucciones (*system prompt*), indica el nombre exacto de cada clave, su tipo y que use `null` cuando el anuncio no dé el dato.
- Pide que responda **solo** con el JSON. Aun así, el modelo puede añadir texto o marcas ```` ```json ````: límpialas antes de usar `json.loads()`.

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.environ.get('OPENROUTER_API_KEY') or getpass('Pega tu clave de OpenRouter: ')
client = OpenAI(base_url='https://openrouter.ai/api/v1', api_key=api_key)
MODELO = 'nvidia/nemotron-3-ultra-550b-a55b:free'   # puedes cambiarlo por otro modelo :free

INSTRUCCIONES = (
    # TODO: escribe las instrucciones para el modelo
    ''
)


def extraer_campos(texto):
    """Devuelve un diccionario con los campos extraídos del anuncio."""
    # TODO: llama al modelo, limpia la respuesta y conviértela con json.loads()
    return {}


# TODO: aplica la función a todos los anuncios y construye el DataFrame
# extraidos = pd.DataFrame([extraer_campos(t) for t in anuncios['texto']])
# pisos = pd.concat([anuncios[['id']], extraidos], axis=1)
# pisos

**Parte B — Tipos y revisión.**
Asigna a cada columna de `pisos` el `dtype` adecuado (recuerda que hay nulos) y **revisa a mano** el resultado contra los textos. Anota los errores que haya cometido el modelo.

In [ ]:
# TODO: dtypes adecuados

# TODO: revisión — compara cada fila con su texto

**Preguntas:**

1. ¿En qué anuncios ha fallado o ha dudado el modelo? ¿Por qué crees que ha pasado?
2. El anuncio A6 no dice el precio. ¿Qué ha devuelto el modelo? ¿Qué debería devolver?
3. Si tuvieras **50.000 anuncios**, ¿usarías este método? Piensa en coste, tiempo, límites de peticiones y en cómo comprobarías la calidad del resultado.

*Escribe aquí tus respuestas:*

---
# Sesión 2 — Entidades, atributos y relaciones

## Ejercicio 2.1 — La biblioteca del instituto

La biblioteca del instituto lleva los préstamos en una **única hoja de cálculo**. Cada fila es un préstamo, pero en ella aparecen también los datos del alumno y del libro.

In [ ]:
prestamos_plano = pd.DataFrame({
    'id_prestamo':      [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    'fecha_prestamo':   ['2026-10-01', '2026-10-01', '2026-10-02', '2026-10-05', '2026-10-05',
                         '2026-10-06', '2026-10-08', '2026-10-08', '2026-10-09', '2026-10-12'],
    'fecha_devolucion': ['2026-10-15', None, '2026-10-09', '2026-10-19', None,
                         '2026-10-20', None, '2026-10-22', None, None],
    'nia':              ['A101', 'A102', 'A101', 'A103', 'A104', 'A102', 'A105', 'A103', 'A101', 'A104'],
    'nombre_alumno':    ['Irene Castro', 'Mario Vílchez', 'Irene Castro', 'Sara Molina', 'Pablo Ruiz',
                         'Mario Vílchez', 'Nerea López', 'Sara Molina', 'Irene Castro', 'Pablo Ruiz'],
    'grupo':            ['GDE', 'GDE', 'GDE', '1DAM', '1DAM', 'GDE', '2DAM', '1DAM', 'GDE', '1DAM'],
    'tutor_grupo':      ['A. de Alarcón', 'A. de Alarcón', 'A. de Alarcón', 'M. Fernández', 'M. Fernández',
                         'A. de Alarcón', 'J. Pérez', 'M. Fernández', 'A. de Alarcón', 'M. Fernández'],
    'isbn':             ['978-84-415-4271-8', '978-84-267-3305-3', '978-84-415-4271-8', '978-84-206-7495-1',
                         '978-84-267-3305-3', '978-84-376-0494-7', '978-84-206-7495-1', '978-84-415-4271-8',
                         '978-84-376-0494-7', '978-84-206-7495-1'],
    'titulo':           ['Python para análisis de datos', 'Aprende Machine Learning', 'Python para análisis de datos',
                         'Romancero gitano', 'Aprende Machine Learning', 'Cien años de soledad',
                         'Romancero Gitano', 'Python para análisis de datos', 'Cien años de soledad',
                         'Romancero gitano'],
    'autor':            ['W. McKinney', 'A. Géron', 'W. McKinney', 'F. García Lorca', 'A. Géron',
                         'G. García Márquez', 'F. García Lorca', 'W. McKinney', 'G. García Márquez',
                         'F. García Lorca'],
    'editorial':        ['Anaya', 'Marcombo', 'Anaya', 'Alianza', 'Marcombo', 'Cátedra', 'Alianza', 'Anaya',
                         'Cátedra', 'Alianza'],
})

prestamos_plano

**Parte A — Entidades y dependencias.**

1. ¿Cuál es la **granularidad** de la tabla? ¿Qué atributo puede ser su clave primaria? Compruébalo con código.
2. Identifica las **entidades** que están «colapsadas» en la tabla y qué atributos pertenecen a cada una.
3. Comprueba con `groupby(...).nunique()` que cada atributo depende de la clave de su entidad (por ejemplo, que cada `nia` tiene siempre el mismo `nombre_alumno`). Uno de los atributos **no cumple** la comprobación por un error de introducción de datos. ¿Cuál? Corrígelo.

In [ ]:
# TODO: granularidad y clave primaria

# TODO: comprobación de dependencias (cada clave → un único valor del atributo)

# TODO: corrige el error de introducción de datos

**Parte B — Normalización.**
Separa la tabla en un DataFrame por entidad (`alumnos`, `grupos`, `libros`, `prestamos`), sin filas duplicadas y con sus claves. Después:

1. Indica la **cardinalidad** de cada relación (1:1, 1:N o N:M).
2. Calcula cuántas **celdas** ocupa la tabla plana y cuántas suman las tablas normalizadas.
3. Reconstruye la tabla plana con `merge()` a partir de las tablas normalizadas y comprueba que obtienes la misma información.

In [ ]:
# TODO: tablas normalizadas
# alumnos   = ...
# grupos    = ...
# libros    = ...
# prestamos = ...

# TODO: celdas antes y después

# TODO: reconstrucción con merge

**Parte C — Preguntas a los datos.**
Responde con las tablas normalizadas:

1. ¿Qué préstamos siguen **sin devolver**? Muestra el nombre del alumno, su grupo y el título del libro.
2. ¿Cuál es el libro más prestado?
3. ¿Cuántos días de media tarda en devolverse un libro (solo los devueltos)?

In [ ]:
# TODO: preguntas 1 a 3

**Diagrama.**
Dibuja el diagrama entidad-relación del esquema normalizado: entidades, atributos, claves primarias y foráneas, y la cardinalidad de cada relación. Puedes hacerlo a mano (y pegar una foto), con [draw.io](https://app.diagrams.net) o con matplotlib como en el Ejemplo 3 de la sesión 2.

*Pega aquí tu diagrama o explica dónde está:*

## Ejercicio 2.2 — Matrículas: integridad referencial y granularidad

Secretaría nos pasa tres tablas del curso. Antes de usarlas, hay que comprobar que son coherentes entre sí.

In [ ]:
alumnado = pd.DataFrame({
    'nia':       ['N01', 'N02', 'N03', 'N04', 'N05', 'N05', 'N06'],
    'nombre':    ['Ana', 'Bruno', 'Carmen', 'David', 'Eva', 'Eva', 'Fran'],
    'localidad': ['Granada', 'Armilla', 'Granada', 'Ogíjares', 'Granada', 'Granada', 'Motril'],
})

modulos = pd.DataFrame({
    'codigo': ['5104', '5109', '5134', '5144', '5149'],
    'nombre': ['ETCDFM', 'DLCD', 'Modelos de aprendizaje automático', 'IA y aprendizaje automático',
               'Componentes software'],
    'horas_semana': [8, 5, 5, 3, 6],
})

matriculas = pd.DataFrame({
    'nia':    ['N01', 'N01', 'N01', 'N02', 'N02', 'N03', 'N03', 'N03', 'N03', 'N04', 'N05', 'N07', 'N02'],
    'codigo': ['5104', '5109', '5134', '5104', '5109', '5104', '5109', '5134', '5149', '5109', '5104', '5109', '5199'],
    'nota':   [7.5, 8.0, 6.5, 5.0, 4.5, 9.0, 9.5, 8.5, 9.0, 6.0, 7.0, 8.0, 5.5],
})

print(f'alumnado: {len(alumnado)} filas · modulos: {len(modulos)} filas · matriculas: {len(matriculas)} filas')

**Parte A — Claves e integridad.**

1. Comprueba si `nia` es una clave primaria válida en `alumnado` y `codigo` en `modulos`. Si no lo es, explica el problema y corrígelo.
2. ¿Cuál es la clave primaria de `matriculas`? Compruébalo.
3. Comprueba la **integridad referencial** de las dos claves foráneas de `matriculas`. Lista las filas huérfanas e indica a qué tabla les falta el registro.
4. ¿Hay algún alumno **sin ninguna matrícula**?

In [ ]:
# TODO: claves primarias

# TODO: integridad referencial

# TODO: alumnado sin matrícula

**Parte B — Tipos de `merge`.**
Combina `matriculas` con `alumnado` (una vez corregido) usando `how='inner'`, `'left'`, `'right'` y `'outer'`. Para cada caso, anota cuántas filas resultan y explica **por qué** salen esas filas. Usa `indicator=True` para ver de qué tabla procede cada fila.

In [ ]:
# TODO: los cuatro tipos de merge

**Parte C — Cuidado con la granularidad.**
Jefatura de estudios pide la **nota media del alumnado de Granada**. Calcúlala de dos formas:

1. Uniendo `matriculas` con `alumnado` y haciendo la media de la columna `nota`.
2. Calculando primero la nota media de **cada alumno** y después la media de esas medias.

¿Salen igual? ¿Por qué? ¿Cuál responde mejor a la pregunta? Explícalo pensando en **qué representa cada fila** en cada cálculo.

In [ ]:
# TODO: las dos formas de calcular la nota media

*Escribe aquí tus respuestas:*

## Ejercicio 2.3 — Documenta una base de datos real: Chinook

[Chinook](https://github.com/lerocha/chinook-database) es una base de datos de ejemplo de una tienda de música digital (artistas, álbumes, canciones, clientes, facturas…). Es una **base de datos relacional real** en SQLite, con varias tablas y claves foráneas.

La celda siguiente la descarga (solo la primera vez) y abre la conexión.

In [ ]:
import urllib.request

URL_CHINOOK = 'https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite'
ruta_chinook = Path('chinook.sqlite')

if not ruta_chinook.exists():
    print('Descargando Chinook...')
    urllib.request.urlretrieve(URL_CHINOOK, ruta_chinook)

con = sqlite3.connect(ruta_chinook)
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name", con)
tablas

**Parte A — Volumetrías y claves.**
Construye un DataFrame `ficha` con **una fila por tabla** y estas columnas:

| tabla | filas | columnas | clave_primaria | claves_foraneas (→ tabla referenciada) |
|---|---|---|---|---|

Pistas:
- `pd.read_sql('SELECT * FROM "Tabla"', con)` carga una tabla entera.
- `PRAGMA table_info("Tabla")` devuelve las columnas; la columna `pk` indica cuáles forman la clave primaria.
- `PRAGMA foreign_key_list("Tabla")` devuelve las claves foráneas.

In [ ]:
# TODO: ficha de volumetrías y claves
# filas = []
# for tabla in tablas['name']:
#     ...
# ficha = pd.DataFrame(filas)
# ficha

**Parte B — Relaciones.**

1. Escribe la lista de relaciones del esquema con su cardinalidad (por ejemplo: `Artist 1:N Album`).
2. Hay dos tablas que resuelven relaciones **N:M**. ¿Cuáles son y qué entidades relacionan?
3. La tabla `Employee` tiene una clave foránea **hacia sí misma**. ¿Qué representa?
4. ¿Cuál es la **granularidad** de `Invoice` y de `InvoiceLine`? ¿Qué cifra obtendrías si sumaras `Total` de `Invoice` después de unirla con `InvoiceLine`? ¿Sería correcta?

*Escribe aquí tus respuestas:*

**Parte C — Preguntas con `merge`.**
Carga las tablas que necesites como DataFrames y responde **con pandas** (no con SQL):

1. Los **5 géneros musicales** que más dinero han facturado (`InvoiceLine.UnitPrice × Quantity`).
2. El **país** con más clientes y el país con más facturación.
3. Comprueba la integridad referencial de `Track.GenreId` y de `InvoiceLine.TrackId`. ¿Hay huérfanos?

In [ ]:
# TODO: preguntas 1 a 3

**Parte D — Diagrama.**
Dibuja el diagrama entidad-relación de Chinook (puedes omitir los atributos que no sean claves). Herramientas: a mano, [draw.io](https://app.diagrams.net) o [dbdiagram.io](https://dbdiagram.io).

*Pega aquí tu diagrama o explica dónde está:*

---
# Sesión 3 — Formatos, carga y repositorios

## Ejercicio 3.1 — Datos abiertos del INE: la población de Granada

El **Instituto Nacional de Estadística** publica sus tablas en CSV. Vamos a usar la tabla **2852 — Población por provincias y sexo** (cifras del padrón):

```
https://www.ine.es/jaxiT3/files/t/es/csv_bdsc/2852.csv
```

Si la descarga directa falla, entra en <https://www.ine.es/jaxiT3/Tabla.htm?t=2852>, pulsa el botón de descarga, elige **CSV: separado por ;** y guarda el fichero en esta carpeta.

**Parte A — Carga ingenua.**
Carga el fichero con `pd.read_csv()` **sin ningún parámetro** y observa el resultado. Describe todos los problemas que veas.

In [ ]:
URL_INE = 'https://www.ine.es/jaxiT3/files/t/es/csv_bdsc/2852.csv'

# TODO: carga sin parámetros
# df_ine_mal = pd.read_csv(URL_INE)
# df_ine_mal.head()

*Problemas detectados en la carga ingenua:*

**Parte B — Carga correcta.**
Vuelve a cargarlo con los parámetros adecuados (`sep`, `thousands`, `decimal`, `encoding`, `na_values`, `dtype`…) para que la columna `Total` sea numérica. Después:

1. La columna `Provincias` mezcla el **código** y el **nombre** de la provincia (por ejemplo, `18 Granada`). Sepárala en dos columnas, `cod_provincia` y `provincia`.
2. Asigna a cada columna su `dtype` adecuado.
3. Muestra la **evolución de la población de Granada** (ambos sexos) en los años disponibles.
4. ¿En qué año tuvo Granada más población? ¿Cuántos habitantes más o menos tiene ahora que el primer año de la serie?

In [ ]:
# TODO: carga correcta

# TODO: separar código y nombre de la provincia

# TODO: dtypes

# TODO: evolución de la población de Granada

**Preguntas:**

1. ¿Cuál es la **granularidad** de la tabla? ¿Qué filas tendrías que quitar para no contar dos veces a la misma población al sumar?
2. ¿Por qué el INE usa el punto como separador de miles y `;` como separador de campos? ¿Qué pasaría si usara `,` como separador de campos?

*Escribe aquí tus respuestas:*

## Ejercicio 3.2 — Caso práctico: la red de calidad del aire

Una red de estaciones de calidad del aire de la provincia de Granada guarda su información en **cinco ficheros de cinco formatos distintos**, porque cada parte la genera un sistema diferente. *(Los datos son simulados.)*

| Fichero | Formato | Contenido |
|---|---|---|
| `aire/estaciones.xml` | XML | Estaciones de medida |
| `aire/mediciones_enero.csv` | CSV | Mediciones diarias de enero |
| `aire/mediciones_febrero.json` | JSON | Mediciones diarias de febrero |
| `aire/contaminantes.xlsx` | Excel (2 hojas) | Contaminantes y su valor límite · índice de calidad del aire |
| `aire/incidencias.db` | SQLite | Incidencias de mantenimiento de las estaciones |

Ejecuta la celda para generar los ficheros.

In [ ]:
# ── Generación de los ficheros (no hace falta que entiendas este código) ─────
carpeta = Path('aire')
carpeta.mkdir(exist_ok=True)
rng = np.random.default_rng(2027)

estaciones = [
    ('GR01', 'Granada Norte', 'Granada', 'tráfico', 37.1969, -3.6121),
    ('GR02', 'Palacio de Congresos', 'Granada', 'fondo urbano', 37.1653, -3.5987),
    ('GR03', 'Armilla', 'Armilla', 'fondo suburbano', 37.1414, -3.6283),
    ('GR04', 'Motril', 'Motril', 'industrial', 36.7471, -3.5179),
]
xml = ['<?xml version="1.0" encoding="UTF-8"?>', '<red nombre="Red provincial de calidad del aire">']
for cod, nom, mun, tipo, lat, lon in estaciones:
    xml.append(f'  <estacion codigo="{cod}">\n    <nombre>{nom}</nombre>\n    <municipio>{mun}</municipio>\n'
               f'    <tipo>{tipo}</tipo>\n    <lat>{lat}</lat>\n    <lon>{lon}</lon>\n  </estacion>')
xml.append('</red>')
(carpeta / 'estaciones.xml').write_text('\n'.join(xml), encoding='utf-8')

def mediciones(mes, dias):
    filas = []
    for cod, *_ in estaciones:
        base = {'GR01': 42, 'GR02': 28, 'GR03': 22, 'GR04': 30}[cod]
        for d in range(1, dias + 1):
            filas.append({
                'estacion': cod, 'fecha': pd.Timestamp(2027, mes, d),
                'no2': round(max(rng.normal(base, 9), 3), 1),
                'pm10': round(max(rng.normal(base * 0.8, 10), 2), 1),
                'o3': round(max(rng.normal(70 - base * 0.6, 12), 5), 1),
            })
    return pd.DataFrame(filas)

ene = mediciones(1, 31)
ene.loc[rng.choice(len(ene), 6, replace=False), 'pm10'] = np.nan     # el analizador falló
ene.loc[17, 'no2'] = -999                                           # código de error del sensor
ene_csv = ene.assign(fecha=ene['fecha'].dt.strftime('%d/%m/%Y')).rename(columns={
    'estacion': 'Estación', 'fecha': 'Fecha', 'no2': 'NO2 (µg/m³)', 'pm10': 'PM10 (µg/m³)', 'o3': 'O3 (µg/m³)'})
ene_csv.to_csv(carpeta / 'mediciones_enero.csv', sep=';', decimal=',', index=False,
               na_rep='-', encoding='latin-1')

feb = mediciones(2, 28)
feb_json = [{'estacion': r.estacion, 'fecha': r.fecha.strftime('%Y-%m-%d'),
             'medidas': {'NO2': r.no2, 'PM10': r.pm10, 'O3': r.o3}} for r in feb.itertuples()]
(carpeta / 'mediciones_febrero.json').write_text(json.dumps(feb_json, ensure_ascii=False), encoding='utf-8')

with pd.ExcelWriter(carpeta / 'contaminantes.xlsx') as xls:
    pd.DataFrame({
        'codigo': ['no2', 'pm10', 'o3'],
        'nombre': ['Dióxido de nitrógeno', 'Partículas < 10 µm', 'Ozono'],
        'unidad': ['µg/m³'] * 3,
        'valor_limite_diario': [50, 45, 100],
    }).to_excel(xls, sheet_name='contaminantes', index=False)
    pd.DataFrame({
        'nivel': ['Buena', 'Razonablemente buena', 'Regular', 'Desfavorable', 'Muy desfavorable'],
        'no2_hasta': [40, 90, 120, 230, 340],
    }).to_excel(xls, sheet_name='indice_calidad', index=False)

ruta_db = carpeta / 'incidencias.db'
ruta_db.unlink(missing_ok=True)
with sqlite3.connect(ruta_db) as c:
    c.execute('CREATE TABLE incidencias (id INTEGER PRIMARY KEY, estacion TEXT, fecha TEXT, tipo TEXT, descripcion TEXT)')
    c.executemany('INSERT INTO incidencias VALUES (?, ?, ?, ?, ?)', [
        (1, 'GR01', '2027-01-18', 'avería', 'Fallo del analizador de NO2'),
        (2, 'GR03', '2027-01-25', 'calibración', 'Calibración programada'),
        (3, 'GR04', '2027-02-10', 'avería', 'Corte de suministro eléctrico'),
        (4, 'GR05', '2027-02-14', 'instalación', 'Montaje de la nueva estación de Baza'),
        (5, 'GR02', '2027-02-21', 'calibración', 'Calibración programada'),
    ])
c.close()

for f in sorted(carpeta.iterdir()):
    print(f'{f.name:28s} {f.stat().st_size:>8,} bytes')

**Parte A — Carga.**
Carga cada fichero con la función `read_*` adecuada y los parámetros necesarios. Abre antes los ficheros de texto (CSV, JSON y XML) con VS Code para ver cómo están escritos: separador, decimales, codificación, cómo se marcan los nulos, estructura…

Al terminar debes tener: `df_estaciones`, `df_enero`, `df_febrero`, `df_contaminantes`, `df_indice` y `df_incidencias`.

In [ ]:
# TODO: estaciones (XML)
# df_estaciones = pd.read_xml(...)

# TODO: mediciones de enero (CSV)
# df_enero = pd.read_csv(...)

# TODO: mediciones de febrero (JSON anidado)
# df_febrero = pd.json_normalize(...)

# TODO: contaminantes e índice de calidad (Excel, dos hojas)

# TODO: incidencias (SQLite)

**Parte B — Integración.**
Une enero y febrero en un único DataFrame `mediciones` con las columnas `estacion`, `fecha`, `no2`, `pm10` y `o3`. Las columnas deben tener **el mismo nombre y el mismo `dtype`** en los dos meses.

Revisa después la calidad de la unión:

1. ¿Cuántos valores nulos hay en cada contaminante? ¿De qué mes vienen?
2. Hay un valor **imposible** escondido en los datos. Encuéntralo, explica qué significa y conviértelo en nulo.
3. Comprueba que no hay mediciones duplicadas (misma estación y misma fecha).

In [ ]:
# TODO: integración de enero y febrero

# TODO: revisión de calidad

**Parte C — Entidades y documentación.**

1. Identifica las **entidades** del sistema, sus **claves primarias** y las **relaciones** entre ellas (con su cardinalidad).
2. Comprueba la **integridad referencial** de `mediciones.estacion` y de `incidencias.estacion` frente a `estaciones`. ¿Qué ocurre con la incidencia 4? ¿Es un error o tiene explicación?
3. Construye una tabla de **volumetrías**: filas, columnas y memoria de cada DataFrame.

In [ ]:
# TODO: integridad referencial

# TODO: volumetrías

*Entidades, claves y relaciones:*

**Parte D — Resultado.**

1. Añade a `mediciones` el **nombre** y el **tipo** de cada estación.
2. Añade una columna `calidad_no2` con el nivel del índice de calidad según el NO2 (hoja `indice_calidad`), como **categórica ordinal**. Pista: `pd.cut()` con los límites de la hoja.
3. Calcula, para cada estación, cuántos días **superó el valor límite diario** de cada contaminante.
4. Guarda el resultado en **Parquet** (`aire/mediciones_integradas.parquet`) y vuelve a leerlo. ¿Se conservan los `dtypes`? ¿Pasaría lo mismo si lo guardaras en CSV?

In [ ]:
# TODO: partes D.1 a D.4

**Preguntas:**

1. Si fueras responsable de esta red, ¿en qué **formato único** guardarías todos estos datos a partir de ahora? Justifícalo.
2. ¿Qué ventajas tiene que las estaciones estén en una tabla aparte, en lugar de repetir su nombre y su tipo en cada medición?

*Escribe aquí tus respuestas:*

## Ejercicio 3.3 — Repositorios de datos: ¿de dónde vienen y qué nos cuentan?

Un buen repositorio no solo da el fichero: también da **metadatos** (quién lo creó, cuándo, con qué licencia, qué significa cada columna…). Sin ellos es difícil saber si un dataset es fiable o si se puede usar.

**Parte A — Investiga.**
Busca un dataset sobre **el mismo tema** en tres repositorios distintos. Elige un tema que te interese (por ejemplo: calidad del aire, accidentes de tráfico, rendimiento académico, consumo eléctrico, turismo…). Al menos uno de los tres debe ser un portal de **datos abiertos públicos** ([datos.gob.es](https://datos.gob.es), [Junta de Andalucía](https://www.juntadeandalucia.es/datosabiertos/portal.html), [INE](https://www.ine.es), [Eurostat](https://ec.europa.eu/eurostat/data/database)…) y otro debe ser [Kaggle](https://www.kaggle.com/datasets), [UCI](https://archive.ics.uci.edu) o [Hugging Face](https://huggingface.co/datasets).

| Aspecto | Dataset 1 | Dataset 2 | Dataset 3 |
|---|---|---|---|
| Nombre y enlace | | | |
| Repositorio | | | |
| Quién lo publica | | | |
| Licencia | | | |
| Fecha de la última actualización | | | |
| Formato(s) de descarga | | | |
| Filas × columnas (aprox.) | | | |
| ¿Describe el significado de cada columna? | | | |
| ¿Explica cómo se recogieron los datos? | | | |
| Naturaleza (estructurado / semi / no estructurado) | | | |

*Edita la tabla en la celda de arriba o responde aquí:*

**Parte B — Carga uno de ellos.**
Carga en pandas uno de los tres datasets (desde su URL, desde el fichero descargado o con la librería del repositorio) y muestra: dimensiones, `dtypes` y las primeras filas. Corrige los `dtypes` que no sean adecuados.

In [ ]:
# TODO: carga y corrección de tipos

**Preguntas:**

1. ¿Cuál de los tres datasets elegirías para un proyecto real? ¿Por qué? Ten en cuenta la licencia, la documentación y la calidad aparente de los datos.
2. ¿Qué metadatos echas en falta en alguno de ellos?
3. ¿Qué diferencia hay entre una licencia **CC BY 4.0** y una **CC BY-NC**? ¿Podrías usar un dataset CC BY-NC para entrenar un modelo que vaya a vender una empresa?

*Escribe aquí tus respuestas:*